# Student Academic Success Prediction — Phase 1
## Problem Definition, EDA, Preprocessing & Feature Engineering

**Dataset:** Open University Learning Analytics Dataset (OULAD)  
**Task:** Classify student `final_result` into: Distinction, Pass, Fail, Withdrawn  
**Source:** https://analyse.kmi.open.ac.uk/open_dataset

---

### 1. Problem Definition

The Open University (UK) collects detailed digital footprints of its distance-learning students. Our goal is to **predict whether a student will pass, fail, earn a distinction, or withdraw** from a module, using:

- **Demographics** — age, gender, region, deprivation index, prior education, disability  
- **Engagement** — daily clicks on the Virtual Learning Environment (VLE)  
- **Assessment performance** — scores, submission timing  
- **Registration behaviour** — how early they enrolled  

This is a **multi-class classification** problem with practical value: early identification of at-risk students enables timely intervention.

#### Prediction Point

In a realistic deployment, this model would be used as an **early-warning system** — making predictions at a specific day into the module, using only the data available up to that point. This means:

- **VLE engagement data** is filtered to only include clicks up to the prediction day  
- **Assessment scores** are filtered to only include submissions made before the prediction day  
- **Demographics and registration** are always available (they exist before the module starts)  

We define a configurable `PREDICTION_DAY` parameter. Setting it to, for example, day 100 means we only use the first ~100 days of behavioural data — simulating a prediction made roughly halfway through a typical 240-day module. This allows us to answer: *"Using only the data available at this point, can we identify who will fail or withdraw?"*

In Phase 2, we will experiment with multiple prediction days to study how accuracy improves as more data becomes available.

---

### Dataset Structure

The OULAD is a **relational** dataset with 7 CSV tables linked by composite keys:

| Table | Rows | Description |
|---|---|---|
| `courses` | 22 | Module-presentation metadata (length) |
| `assessments` | 206 | Assessment schedule & weights |
| `vle` | 6 364 | VLE page/material catalogue |
| `studentInfo` | 32 593 | Demographics + **target** (`final_result`) |
| `studentRegistration` | 32 593 | Registration & un-registration dates |
| `studentAssessment` | 173 912 | Submitted scores per assessment |
| `studentVle` | ~10.6 M | Daily click logs per student per VLE page |

**Join keys:** `code_module` + `code_presentation` link courses/assessments/vle; adding `id_student` links to student tables.

## 2. Setup & Data Loading

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['figure.dpi'] = 100

# ============================================================
# PREDICTION DAY — configurable cutoff
# Only VLE clicks and assessment submissions up to this day
# are included. Set to None to use all available data.
# ============================================================
PREDICTION_DAY = None  # None = use all data (retrospective); try 100, 150, etc. in Phase 2

if PREDICTION_DAY is not None:
    print(f">>> PREDICTION MODE: Using only data up to day {PREDICTION_DAY}")
else:
    print(">>> RETROSPECTIVE MODE: Using all available data (no time cutoff)")
print("Libraries loaded.")

In [ ]:
courses = pd.read_csv('datasets/courses.csv')
assessments = pd.read_csv('datasets/assessments.csv')
vle = pd.read_csv('datasets/vle.csv')
student_info = pd.read_csv('datasets/studentInfo.csv')
student_reg = pd.read_csv('datasets/studentRegistration.csv')
student_assess = pd.read_csv('datasets/studentAssessment.csv')
student_vle = pd.read_csv('datasets/studentVle.csv')

tables = {
    'courses': courses, 'assessments': assessments, 'vle': vle,
    'studentInfo': student_info, 'studentRegistration': student_reg,
    'studentAssessment': student_assess, 'studentVle': student_vle
}

for name, df in tables.items():
    print(f"{name:25s}  shape={str(df.shape):15s}  missing(?)={int((df == '?').sum().sum()):>6}")

The `?` character is used as a placeholder for missing values across several tables. We will handle these during preprocessing. The largest table by far is `studentVle` with over 10 million rows of daily click logs.

In [ ]:
# Summary of missing values ('?') across all tables
print('Missing values summary ("?" placeholders):')
print(f'  studentInfo.imd_band:              {(student_info.imd_band == "?").sum():>5} / {len(student_info)} ({(student_info.imd_band == "?").mean()*100:.1f}%)')
print(f'  studentAssessment.score:            {(student_assess.score == "?").sum():>5} / {len(student_assess)} ({(student_assess.score == "?").mean()*100:.2f}%)')
print(f'  studentRegistration.date_reg:       {(student_reg.date_registration == "?").sum():>5} / {len(student_reg)} ({(student_reg.date_registration == "?").mean()*100:.2f}%)')
print(f'  studentRegistration.date_unreg:     {(student_reg.date_unregistration == "?").sum():>5} / {len(student_reg)} (expected — means student stayed)')
print(f'  assessments.date:                   {(assessments.date == "?").sum():>5} / {len(assessments)} (exams have no fixed date)')
print(f'  vle.week_from/week_to:              {(vle.week_from == "?").sum():>5} / {len(vle)} (materials with no scheduled week)')

### Feature Descriptions per Table

Before starting the EDA, we examine the columns, data types, and first few rows of each table.

In [ ]:
print('=' * 70)
print('COURSES (22 rows) — one row per module-presentation')
print('=' * 70)
print(courses.dtypes.to_string())
print(courses.head(3).to_string())
print(f'\n  code_module:                  module identifier (7 modules: {sorted(courses.code_module.unique())})')
print(f'  code_presentation:            semester (B=February, J=October) + year')
print(f'  module_presentation_length:   duration in days ({courses.module_presentation_length.min()}–{courses.module_presentation_length.max()})')

In [ ]:
print('=' * 70)
print('STUDENT INFO (32,593 rows) — one row per student-module registration')
print('=' * 70)
print(student_info.dtypes.to_string())
print('\n', student_info.head(3).to_string())
print(f'\n  id_student:           unique student ID ({student_info.id_student.nunique()} unique students)')
print(f'  gender:               {sorted(student_info.gender.unique())}')
print(f'  region:               {student_info.region.nunique()} UK regions')
print(f'  highest_education:    {sorted(student_info.highest_education.unique())}')
print(f'  imd_band:             Index of Multiple Deprivation (socioeconomic decile)')
print(f'  age_band:             {sorted(student_info.age_band.unique())}')
print(f'  num_of_prev_attempts: times student previously attempted this module (0–{student_info.num_of_prev_attempts.max()})')
print(f'  studied_credits:      total credits currently studying ({student_info.studied_credits.min()}–{student_info.studied_credits.max()})')
print(f'  disability:           {sorted(student_info.disability.unique())}')
print(f'  final_result:         TARGET — {sorted(student_info.final_result.unique())}')

In [ ]:
print('=' * 70)
print('ASSESSMENTS (206 rows) — assessment schedule per module-presentation')
print('=' * 70)
print(assessments.dtypes.to_string())
print('\n', assessments.head(3).to_string())
print(f'\n  id_assessment:    unique assessment ID')
print(f'  assessment_type:  {sorted(assessments.assessment_type.unique())} (TMA=Tutor Marked, CMA=Computer Marked)')
print(f'  date:             submission deadline in days from module start (? for exams)')
print(f'  weight:           percentage weight of the assessment')

In [ ]:
print('=' * 70)
print('STUDENT ASSESSMENT (173,912 rows) — one row per student submission')
print('=' * 70)
print(student_assess.dtypes.to_string())
print('\n', student_assess.head(3).to_string())
print(f'\n  id_assessment:    links to assessments table')
print(f'  id_student:       links to studentInfo')
print(f'  date_submitted:   day the student submitted (relative to module start)')
print(f'  is_banked:        1 if result transferred from previous attempt')
print(f'  score:            0–100 (? = missing)')

In [ ]:
print('=' * 70)
print('STUDENT REGISTRATION (32,593 rows) — registration dates per student-module')
print('=' * 70)
print(student_reg.dtypes.to_string())
print('\n', student_reg.head(3).to_string())
print(f'\n  date_registration:    days relative to module start (negative = before start)')
print(f'  date_unregistration:  day student unregistered (? = did not unregister)')

In [ ]:
print('=' * 70)
print('VLE (6,364 rows) — catalogue of VLE materials per module')
print('=' * 70)
print(vle.dtypes.to_string())
print('\n', vle.head(3).to_string())
print(f'\n  id_site:         unique material/page ID')
print(f'  activity_type:   {sorted(vle.activity_type.unique())}')
print(f'  week_from/to:    scheduled availability window (? = unscheduled)')

In [ ]:
print('=' * 70)
print(f'STUDENT VLE ({len(student_vle):,} rows) — daily click logs per student per VLE page')
print('=' * 70)
print(student_vle.dtypes.to_string())
print('\n', student_vle.head(3).to_string())
print(f'\n  id_site:     links to vle table')
print(f'  date:        day of interaction (relative to module start)')
print(f'  sum_click:   number of clicks on that page on that day')

The tables are connected through composite keys: `code_module` + `code_presentation` links courses, assessments, and VLE materials. Adding `id_student` links to all student-level tables.

## 3. Exploratory Data Analysis

We analyse each table individually, then examine relationships with the target variable `final_result`.

*Note: The EDA is performed on the **full dataset** (no time cutoff) to understand all available patterns. The time-based filtering is applied during preprocessing (Section 4) based on the `PREDICTION_DAY` parameter.*

### 3.1 Courses and Module Difficulty

In [ ]:
print(courses.to_string(index=False))
print(f"\nModules: {courses.code_module.nunique()}, Presentations: {courses.shape[0]}")
print(f"Module length range: {courses.module_presentation_length.min()}–{courses.module_presentation_length.max()} days")

In [ ]:
order = ['Distinction', 'Pass', 'Fail', 'Withdrawn']
colors = ['#2ecc71', '#3498db', '#e74c3c', '#f39c12']

fig, ax = plt.subplots(figsize=(12, 5))
ct = pd.crosstab(student_info['code_module'], student_info['final_result'], normalize='index') * 100
ct = ct.reindex(columns=order)
ct.plot.bar(stacked=True, ax=ax, color=colors, edgecolor='white', linewidth=0.5)
ax.set_title('Outcome Distribution by Module')
ax.set_ylabel('Percentage'); ax.set_xlabel('Module')
ax.legend(title='', loc='upper right')
ax.tick_params(axis='x', rotation=0)
plt.tight_layout()
plt.show()

**Module difficulty varies considerably.** Some modules (e.g., DDD, CCC) have noticeably higher failure and withdrawal rates than others.

#### Temporal Stability Across Presentations

The dataset spans four semesters (2013B, 2013J, 2014B, 2014J). We check whether outcome distributions are stable across time.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
ct_time = pd.crosstab(student_info['code_presentation'], student_info['final_result'], normalize='index') * 100
ct_time = ct_time.reindex(columns=order)
ct_time.plot.bar(stacked=True, ax=ax, color=colors, edgecolor='white', linewidth=0.5)
ax.set_title('Outcome Distribution by Semester')
ax.set_ylabel('Percentage'); ax.set_xlabel('Semester')
ax.legend(title='', loc='upper right')
ax.tick_params(axis='x', rotation=0)
plt.tight_layout()
plt.show()

**The data shows a temporal trend:** withdrawal rates increased from 2013 (27-29%) to 2014 (33-34%), while failure rates decreased slightly. We note this as a limitation — a model trained on 2013 data might not generalise perfectly to 2014. For this project, we pool all semesters together but acknowledge that a time-aware split could be explored in future work.

### 3.2 Target Variable — `final_result`

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

counts = student_info.final_result.value_counts().reindex(order)
counts.plot.bar(ax=axes[0], color=colors, edgecolor='black')
axes[0].set_title('Class Distribution'); axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=0)
for i, v in enumerate(counts):
    axes[0].text(i, v + 200, f'{v}', ha='center', fontsize=9)

counts.plot.pie(ax=axes[1], autopct='%1.1f%%', colors=colors, startangle=90)
axes[1].set_ylabel(''); axes[1].set_title('Class Proportions')
plt.tight_layout()
plt.show()

print("\nClass balance:")
print((counts / counts.sum() * 100).round(1).to_string())

**Key finding — Class Imbalance:** Pass (37.9%), Withdrawn (31.2%), Fail (21.6%), Distinction (9.3%). Over **52% of students either fail or withdraw**. We will address this in Phase 2 through class weighting or oversampling.

### 3.3 Demographics vs Outcome

We examine how each demographic feature relates to the target, supplementing visual analysis with **Chi-square tests** to confirm significance.

#### 3.3.1 Gender, Age, Education, and Disability

In [ ]:
demo_cols = ['gender', 'age_band', 'highest_education', 'disability']

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.ravel()
for i, col in enumerate(demo_cols):
    ct = pd.crosstab(student_info[col], student_info['final_result'], normalize='index') * 100
    ct = ct.reindex(columns=order)
    ct.plot.bar(stacked=True, ax=axes[i], color=colors, edgecolor='white', linewidth=0.5)
    axes[i].set_title(f'Outcome by {col}'); axes[i].set_ylabel('Percentage')
    axes[i].legend(title='', fontsize=7, loc='upper right')
    axes[i].tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
print("Chi-square tests of independence (demographic feature vs final_result):")
print("-" * 65)
for col in demo_cols:
    ct = pd.crosstab(student_info[col], student_info['final_result'])
    chi2, p, dof, _ = stats.chi2_contingency(ct)
    sig = "***" if p < 0.001 else "**" if p < 0.01 else "*" if p < 0.05 else "ns"
    print(f"  {col:25s}  χ²={chi2:>8.1f}  dof={dof}  p={p:.2e}  {sig}")

**Findings:**
- **Gender** is statistically significant (p < 0.001) but effect size is negligible (~1-2pp difference).
- **Age**: students 55+ earn Distinction at 19% vs 8.1% for 0-35.
- **Highest education** is the strongest demographic predictor: postgraduates 28.1% Distinction vs no-quals 4.6%.
- **Disability** correlates with higher withdrawal (39.3% vs 30.3%).

#### 3.3.2 Region

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))
ct = pd.crosstab(student_info['region'], student_info['final_result'], normalize='index') * 100
ct = ct.reindex(columns=order)
ct.plot.bar(stacked=True, ax=ax, color=colors, edgecolor='white', linewidth=0.5)
ax.set_title('Outcome by Region'); ax.set_ylabel('Percentage')
ax.legend(title='', loc='upper right')
ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.show()

**Region:** Moderate variation, less dramatic than education or age. May overlap with IMD band.

#### 3.3.3 IMD Band (Index of Multiple Deprivation)

Lower percentages = **more deprived** areas. We exclude the 1,111 students (~3.4%) with missing IMD values.

In [ ]:
si_imd = student_info[student_info.imd_band != '?'].copy()
imd_order = ['0-10%','10-20','20-30%','30-40%','40-50%','50-60%','60-70%','70-80%','80-90%','90-100%']
fig, ax = plt.subplots(figsize=(12, 5))
ct = pd.crosstab(pd.Categorical(si_imd['imd_band'], categories=imd_order, ordered=True),
                 si_imd['final_result'], normalize='index') * 100
ct = ct.reindex(columns=order)
ct.plot.bar(stacked=True, ax=ax, color=colors, edgecolor='white', linewidth=0.5)
ax.set_title('Outcome by IMD Band (0-10% = most deprived)')
ax.set_ylabel('Percentage')
ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.show()

**Clear socioeconomic gradient:** most deprived (0-10%) → 37.2% withdrawal, 5.1% distinction; least deprived (90-100%) → 25.9% withdrawal, 14.1% distinction — nearly a **3x difference** in distinction rate.

#### 3.3.4 Studied Credits and Previous Attempts

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

student_info.boxplot(column='studied_credits', by='final_result', ax=axes[0])
axes[0].set_title('Studied Credits by Outcome')
axes[0].set_xlabel(''); axes[0].set_ylabel('Credits')
plt.sca(axes[0]); plt.xticks(range(1,5), order)

student_info.boxplot(column='num_of_prev_attempts', by='final_result', ax=axes[1])
axes[1].set_title('Previous Attempts by Outcome')
axes[1].set_xlabel(''); axes[1].set_ylabel('Attempts')
plt.sca(axes[1]); plt.xticks(range(1,5), order)
plt.suptitle('')
plt.tight_layout()
plt.show()

for col in ['studied_credits', 'num_of_prev_attempts']:
    groups = [g[col].values for _, g in student_info.groupby('final_result')]
    stat, p = stats.kruskal(*groups)
    print(f"Kruskal-Wallis {col} ~ final_result: H={stat:.1f}, p={p:.2e}")

**Studied Credits:** Median is 60 across all groups. Outliers up to 655 credits are kept (may indicate overloaded students). **Previous Attempts:** Fail students average 0.25 vs 0.06 for Distinction. Both are statistically significant (Kruskal-Wallis).

### 3.4 Assessments

In [ ]:
assessments_clean = assessments.copy()
assessments_clean['date'] = pd.to_numeric(assessments_clean['date'].replace('?', np.nan))

print("Assessment types:")
print(assessments_clean.assessment_type.value_counts())
print(f"\nAssessments with no fixed date (Exams): {assessments_clean.date.isna().sum()}")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
assessments_clean.assessment_type.value_counts().plot.bar(ax=axes[0], color='steelblue', edgecolor='black')
axes[0].set_title('Assessment Type Counts'); axes[0].tick_params(axis='x', rotation=0)

assessments_clean.boxplot(column='weight', by='assessment_type', ax=axes[1])
axes[1].set_title('Weight by Assessment Type')
axes[1].set_xlabel(''); axes[1].set_ylabel('Weight (%)')
plt.suptitle('')
plt.tight_layout()
plt.show()

**Two separate grading pools:** TMA+CMA weights sum to 100% (coursework), exams carry a separate 100%. We compute `weighted_coursework` and `exam_score` as separate features.

### 3.5 Student Assessment Scores

In [ ]:
sa = student_assess.copy()
sa['score'] = pd.to_numeric(sa['score'].replace('?', np.nan))

print(f"Total submissions: {len(sa)}")
print(f"Missing scores ('?'): {sa.score.isna().sum()}")
print(f"Banked assessments: {sa.is_banked.sum()}")
print(f"Scores = 0: {(sa.score == 0).sum()}")
print(f"\nScore statistics:")
print(sa.score.describe().round(1))

fig, ax = plt.subplots(figsize=(10, 4))
sa.score.dropna().hist(bins=50, ax=ax, color='steelblue', edgecolor='white')
ax.axvline(sa.score.median(), color='red', linestyle='--', label=f'Median={sa.score.median():.0f}')
ax.set_title('Distribution of Assessment Scores')
ax.set_xlabel('Score'); ax.set_ylabel('Frequency'); ax.legend()
plt.tight_layout()
plt.show()

**Zero scores:** 329 submissions have score = 0. These are overwhelmingly from Fail (153) and Withdrawn (86) students, split across TMA (185) and CMA (141). We treat these as **genuine zero scores** (non-submissions graded as 0) rather than missing data.

In [ ]:
sa_with_info = sa.merge(
    assessments[['id_assessment','code_module','code_presentation','assessment_type']],
    on='id_assessment'
).merge(
    student_info[['code_module','code_presentation','id_student','final_result']],
    on=['code_module','code_presentation','id_student'], how='inner'
)

sa_coursework = sa_with_info[sa_with_info.assessment_type.isin(['TMA','CMA'])]
sa_exam = sa_with_info[sa_with_info.assessment_type == 'Exam']

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
sa_with_info.boxplot(column='score', by='final_result', ax=axes[0])
axes[0].set_title('All Assessments'); axes[0].set_xlabel(''); axes[0].set_ylabel('Score')
plt.sca(axes[0]); plt.xticks(range(1,5), order)

sa_coursework.boxplot(column='score', by='final_result', ax=axes[1])
axes[1].set_title('Coursework (TMA + CMA)'); axes[1].set_xlabel(''); axes[1].set_ylabel('Score')
plt.sca(axes[1]); plt.xticks(range(1,5), order)

sa_exam.boxplot(column='score', by='final_result', ax=axes[2])
axes[2].set_title('Exam Only'); axes[2].set_xlabel(''); axes[2].set_ylabel('Score')
plt.sca(axes[2]); plt.xticks(range(1,5), order)
plt.suptitle('')
plt.tight_layout()
plt.show()

print('Mean COURSEWORK score by outcome:')
print(sa_coursework.groupby('final_result')['score'].mean().reindex(order).round(1))
print('\nMean EXAM score by outcome:')
print(sa_exam.groupby('final_result')['score'].mean().reindex(order).round(1))

**Scores clearly separate outcomes.** Coursework: Distinction 88.5, Pass 77.2, Fail 65.4, Withdrawn 66.1. Exam gap is much larger: Distinction 92.8 vs Fail 34.1 (60-point gap).

**Circularity warning:** Assessment scores largely *determine* `final_result`, not just predict it. Including scores gives high accuracy but is not useful for early warning — the scores only exist after the work is graded. The `PREDICTION_DAY` parameter handles this naturally: at early prediction days, few or no assessment scores are available, forcing the model to rely on engagement and demographics. At late prediction days, scores become available and accuracy improves, but the practical intervention window shrinks.

### 3.6 Registration Behaviour

In [ ]:
sr = student_reg.copy()
sr['date_registration'] = pd.to_numeric(sr['date_registration'].replace('?', np.nan))
sr_info = sr.merge(student_info[['code_module','code_presentation','id_student','final_result']],
                   on=['code_module','code_presentation','id_student'])

fig, ax = plt.subplots(figsize=(10, 5))
sr_info.boxplot(column='date_registration', by='final_result', ax=ax)
ax.set_title('Registration Date by Outcome (0 = module start)')
ax.set_xlabel(''); ax.set_ylabel('Days relative to start')
plt.xticks(range(1,5), order)
plt.suptitle('')
plt.tight_layout()
plt.show()

**Registration timing** is surprisingly similar across groups (50-67 days before start). Not a strong predictor.

**Data leakage note:** `date_unregistration` is excluded — 99.1% of Withdrawn students un-registered, directly encoding the outcome.

### 3.7 VLE Engagement

In [ ]:
print("Aggregating ~10M VLE click rows — this may take a minute...")

svle_agg = student_vle.groupby(['code_module','code_presentation','id_student']).agg(
    total_clicks=('sum_click', 'sum'),
    active_days=('date', 'nunique'),
    distinct_pages=('id_site', 'nunique')
).reset_index()

print(f"Aggregated to {len(svle_agg)} student-module rows.")
print(svle_agg[['total_clicks','active_days','distinct_pages']].describe().round(1))

In [ ]:
svle_info = svle_agg.merge(
    student_info[['code_module','code_presentation','id_student','final_result']],
    on=['code_module','code_presentation','id_student']
)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for i, col in enumerate(['total_clicks','active_days','distinct_pages']):
    svle_info.boxplot(column=col, by='final_result', ax=axes[i], showfliers=False)
    axes[i].set_title(col.replace('_',' ').title() + ' by Outcome')
    axes[i].set_xlabel('')
    plt.sca(axes[i]); plt.xticks(range(1,5), order)
plt.suptitle('')
plt.tight_layout()
plt.show()

print("Median engagement by outcome:")
print(svle_info.groupby('final_result')[['total_clicks','active_days','distinct_pages']].median().reindex(order).round(0))

stat, p = stats.kruskal(*[g['total_clicks'].values for _, g in svle_info.groupby('final_result')])
print(f"\nKruskal-Wallis total_clicks ~ final_result: H={stat:.1f}, p={p:.2e}")

**VLE engagement is the strongest predictor.** Median total clicks: Distinction 1,896 vs Withdrawn 222 (**9x**). Active days: 103 vs 15 (**7x**). Distinct pages: 82 vs 25 (**3x**). Kruskal-Wallis confirms extreme significance. ~3,365 students (10.3%) had zero VLE activity — almost all failed or withdrew.

#### VLE Activity Type Breakdown

In [ ]:
vle_types = student_vle.merge(vle[['id_site','activity_type']], on='id_site')
type_clicks = vle_types.groupby('activity_type')['sum_click'].sum().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(12, 5))
type_clicks.head(10).plot.bar(ax=ax, color='steelblue', edgecolor='black')
ax.set_title('Top 10 VLE Activity Types by Total Clicks')
ax.set_ylabel('Total Clicks')
ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.show()

**`oucontent`** dominates, followed by **`forumng`** (forums) and **`subpage`**. We create per-activity-type click features.

### 3.8 Correlation Heatmap

In [ ]:
numeric_preview = student_info[['code_module','code_presentation','id_student',
                                 'num_of_prev_attempts','studied_credits','final_result']].merge(
    svle_agg, on=['code_module','code_presentation','id_student'], how='left'
).merge(
    sa.groupby('id_student').agg(mean_score=('score','mean'), score_std=('score','std')).reset_index(),
    on='id_student', how='left'
)

num_cols = ['num_of_prev_attempts','studied_credits','total_clicks','active_days',
            'distinct_pages','mean_score','score_std']

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(numeric_preview[num_cols].corr(), annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, ax=ax, square=True)
ax.set_title('Correlation Matrix — Numeric Features')
plt.tight_layout()
plt.show()

**Key correlations:** VLE metrics (total_clicks, active_days, distinct_pages) are highly intercorrelated (r > 0.7). mean_score has moderate positive correlation with engagement. studied_credits and score_std contribute independent information.

*Note: mean_score here is a preliminary aggregation. Final features split scores into weighted_coursework and exam_score.*

### 3.9 Student Overlap Across Modules

In [ ]:
modules_per_student = student_info.groupby('id_student').size()

print('Module-presentations per student:')
print(modules_per_student.describe().round(1))
print(f'\nStudents in 1 module:  {(modules_per_student == 1).sum()} ({(modules_per_student == 1).mean()*100:.1f}%)')
print(f'Students in 2+ modules: {(modules_per_student > 1).sum()} ({(modules_per_student > 1).mean()*100:.1f}%)')

fig, ax = plt.subplots(figsize=(8, 4))
modules_per_student.value_counts().sort_index().plot.bar(ax=ax, color='steelblue', edgecolor='black')
ax.set_title('Module-Presentations per Student')
ax.set_xlabel('Number of modules'); ax.set_ylabel('Number of students')
ax.tick_params(axis='x', rotation=0)
plt.tight_layout()
plt.show()

**12.3% of students appear in multiple modules** (up to 5). Requires **group-aware train-test split** by `id_student` in Phase 2.

---

### 3.10 EDA Summary

| Finding | Strength | Implication |
|---|---|---|
| VLE engagement differs 7-9x between Distinction and Withdrawn | Very Strong | Most powerful early-warning feature |
| Assessment scores separate all four classes | Strong | Useful but circular — determines the target |
| Submission behaviour (lateness, rate) correlates with outcomes | Strong | Behavioural signal available at prediction time |
| Higher education → better outcomes | Strong | Demographics contribute even without engagement |
| IMD deprivation → higher failure/withdrawal | Moderate | Socioeconomic factor |
| Older students perform better | Moderate | Age captures maturity/motivation |
| Module difficulty varies | Moderate | Justifies module encoding |
| Withdrawal rate increased 2013→2014 | Moderate | Temporal instability, noted as limitation |
| Gender has negligible effect size | Weak | Unlikely to add value |
| Registration timing similar across groups | Weak | Not a strong predictor |
| 12.3% students in multiple modules | Important | Requires group-aware split |
| Un-registration directly encodes Withdrawal | Leakage | Excluded from features |

## 4. Preprocessing

We now clean, merge, and prepare the dataset for modeling. We handle missing values in each table **before** merging, so we can distinguish between genuinely missing values (`?` placeholders) and NaN values introduced by left joins (meaning "no matching data").

**Time-based filtering:** If `PREDICTION_DAY` is set, we filter VLE clicks and assessment submissions to only include data up to that day. This simulates having only the information available at the prediction point.

### 4.1 Clean Missing Values in Each Table

In [ ]:
# --- studentInfo: base table ---
df = student_info.copy()
df = df.replace('?', np.nan)

# imd_band: fill with mode
print(f"imd_band missing: {df.imd_band.isna().sum()} ({df.imd_band.isna().mean()*100:.1f}%)")
imd_mode = df['imd_band'].mode()[0]
df['imd_band'] = df['imd_band'].fillna(imd_mode)
print(f"Filled with mode: '{imd_mode}'")
print(f"\nBase table shape: {df.shape}")

In [ ]:
# --- studentRegistration: clean ---
sr_clean = student_reg.copy().replace('?', np.nan)
sr_clean['date_registration'] = pd.to_numeric(sr_clean['date_registration'])
sr_clean['date_unregistration'] = pd.to_numeric(sr_clean['date_unregistration'], errors='coerce')
sr_clean['date_registration'] = sr_clean.groupby(['code_module','code_presentation'])['date_registration'].transform(
    lambda x: x.fillna(x.median())
)
print(f"Registration missing after per-module median fill: {sr_clean.date_registration.isna().sum()}")

# --- studentAssessment: clean scores ---
sa_clean = student_assess.copy()
sa_clean['score'] = pd.to_numeric(sa_clean['score'].replace('?', np.nan))
print(f"Assessment missing scores: {sa_clean.score.isna().sum()} out of {len(sa_clean)}")

# --- assessments: clean dates ---
assess_clean = assessments.copy()
assess_clean['date'] = pd.to_numeric(assess_clean['date'].replace('?', np.nan))
print(f"Assessments with no fixed date (Exams): {assess_clean.date.isna().sum()}")

### 4.2 Apply Time-Based Filtering

If `PREDICTION_DAY` is set, we filter time-dependent data to only include records up to that day. Demographics and registration data are always available (they exist before the module starts).

In [ ]:
# --- Filter VLE clicks to only include up to PREDICTION_DAY ---
if PREDICTION_DAY is not None:
    student_vle_filtered = student_vle[student_vle['date'] <= PREDICTION_DAY].copy()
    print(f"VLE clicks filtered to day <= {PREDICTION_DAY}: {len(student_vle_filtered):,} / {len(student_vle):,} rows "
          f"({len(student_vle_filtered)/len(student_vle)*100:.1f}%)")
else:
    student_vle_filtered = student_vle.copy()
    print(f"No time filter applied — using all {len(student_vle_filtered):,} VLE rows")

# --- Filter assessment submissions to only include up to PREDICTION_DAY ---
if PREDICTION_DAY is not None:
    sa_filtered = sa_clean[sa_clean['date_submitted'] <= PREDICTION_DAY].copy()
    print(f"Assessment submissions filtered to day <= {PREDICTION_DAY}: {len(sa_filtered):,} / {len(sa_clean):,} rows "
          f"({len(sa_filtered)/len(sa_clean)*100:.1f}%)")
else:
    sa_filtered = sa_clean.copy()
    print(f"No time filter applied — using all {len(sa_filtered):,} assessment rows")

### 4.3 Merge Registration Data

In [ ]:
df = df.merge(
    sr_clean[['code_module','code_presentation','id_student','date_registration']],
    on=['code_module','code_presentation','id_student'], how='left'
)
print(f"After registration merge: {df.shape}")

### 4.4 Merge Assessment Features

We aggregate assessment data using the (possibly filtered) submissions.

In [ ]:
# Merge assessment metadata
sa_full = sa_filtered.merge(
    assess_clean[['id_assessment','code_module','code_presentation','date','weight','assessment_type']],
    on='id_assessment'
)
sa_full['days_early'] = sa_full['date'] - sa_full['date_submitted']

# Aggregate per student-module
assess_agg = sa_full.groupby(['code_module','code_presentation','id_student']).agg(
    mean_score=('score', 'mean'),
    std_score=('score', 'std'),
    min_score=('score', 'min'),
    max_score=('score', 'max'),
    num_submissions=('score', 'count'),
    mean_days_early=('days_early', 'mean'),
    pct_late=('days_early', lambda x: (x < 0).mean() * 100)
).reset_index()

# std_score: NaN for single-submission students → 0 (no variability)
assess_agg['std_score'] = assess_agg['std_score'].fillna(0)

# Separate coursework and exam scores
def compute_scores(g):
    mask = g['score'].notna() & g['weight'].notna()
    if mask.sum() == 0:
        return pd.Series({'weighted_coursework': np.nan, 'exam_score': np.nan})
    valid = g.loc[mask]
    cw = valid[valid['assessment_type'].isin(['TMA', 'CMA'])]
    cw_score = np.average(cw['score'], weights=cw['weight']) if len(cw) > 0 and cw['weight'].sum() > 0 else np.nan
    ex = valid[valid['assessment_type'] == 'Exam']
    ex_score = ex['score'].mean() if len(ex) > 0 else np.nan
    return pd.Series({'weighted_coursework': cw_score, 'exam_score': ex_score})

score_components = sa_full.groupby(['code_module','code_presentation','id_student']).apply(compute_scores).reset_index()
assess_agg = assess_agg.merge(score_components, on=['code_module','code_presentation','id_student'], how='left')

# Submission rate
expected = assess_clean.groupby(['code_module','code_presentation']).size().reset_index(name='expected_assessments')
assess_agg = assess_agg.merge(expected, on=['code_module','code_presentation'], how='left')
assess_agg['submission_rate'] = assess_agg['num_submissions'] / assess_agg['expected_assessments']

# Submission flags
assess_agg['has_submitted_coursework'] = score_components['weighted_coursework'].notna().astype(int)
assess_agg['has_sat_exam'] = score_components['exam_score'].notna().astype(int)

df = df.merge(assess_agg, on=['code_module','code_presentation','id_student'], how='left')
print(f"After assessment merge: {df.shape}")
print(f"Students with no assessment data: {df.mean_score.isna().sum()}")

### 4.5 Merge VLE Engagement Features

We aggregate VLE data using the (possibly filtered) click logs.

In [ ]:
# Aggregate filtered VLE data
svle_agg_filtered = student_vle_filtered.groupby(['code_module','code_presentation','id_student']).agg(
    total_clicks=('sum_click', 'sum'),
    active_days=('date', 'nunique'),
    distinct_pages=('id_site', 'nunique')
).reset_index()

df = df.merge(svle_agg_filtered, on=['code_module','code_presentation','id_student'], how='left')

# Per-activity-type click counts
vle_with_type = student_vle_filtered.merge(vle[['id_site','activity_type']], on='id_site')
top_types = ['oucontent','forumng','subpage','homepage','resource','quiz','url','ouwiki']
for atype in top_types:
    tmp = vle_with_type[vle_with_type.activity_type == atype].groupby(
        ['code_module','code_presentation','id_student']
    )['sum_click'].sum().reset_index(name=f'clicks_{atype}')
    df = df.merge(tmp, on=['code_module','code_presentation','id_student'], how='left')

# First click date
first_click = student_vle_filtered.groupby(['code_module','code_presentation','id_student'])['date'].min().reset_index(name='first_click_date')
df = df.merge(first_click, on=['code_module','code_presentation','id_student'], how='left')

# Engagement trend: first half vs second half (relative to PREDICTION_DAY or module length)
cutoff_day = PREDICTION_DAY if PREDICTION_DAY is not None else None
svle_with_length = student_vle_filtered.merge(
    courses[['code_module','code_presentation','module_presentation_length']],
    on=['code_module','code_presentation']
)
if cutoff_day is not None:
    svle_with_length['is_first_half'] = svle_with_length['date'] <= (cutoff_day / 2)
else:
    svle_with_length['is_first_half'] = svle_with_length['date'] <= (svle_with_length['module_presentation_length'] / 2)

clicks_first = svle_with_length[svle_with_length.is_first_half].groupby(
    ['code_module','code_presentation','id_student'])['sum_click'].sum().reset_index(name='clicks_first_half')
clicks_second = svle_with_length[~svle_with_length.is_first_half].groupby(
    ['code_module','code_presentation','id_student'])['sum_click'].sum().reset_index(name='clicks_second_half')

df = df.merge(clicks_first, on=['code_module','code_presentation','id_student'], how='left')
df = df.merge(clicks_second, on=['code_module','code_presentation','id_student'], how='left')

print(f"After VLE merge: {df.shape}")

### 4.6 Merge Course Length

In [ ]:
df = df.merge(courses, on=['code_module','code_presentation'], how='left')
print(f"After course merge: {df.shape}")

### 4.7 Handle Post-Merge Missing Values

After the left joins, NaN values represent students with **no matching data** — no VLE activity or no assessment submissions. These are filled with 0 (absence of activity is meaningful, not missing).

In [ ]:
assess_cols = ['mean_score','std_score','min_score','max_score','num_submissions',
               'mean_days_early','pct_late','weighted_coursework','exam_score',
               'submission_rate','has_submitted_coursework','has_sat_exam','expected_assessments']
vle_cols = ['total_clicks','active_days','distinct_pages','first_click_date',
            'clicks_first_half','clicks_second_half'] + [c for c in df.columns if c.startswith('clicks_') and c not in ['clicks_first_half','clicks_second_half']]

print(f"Students with no assessment activity: {df['mean_score'].isna().sum()} ({df['mean_score'].isna().mean()*100:.1f}%)")
print(f"Students with no VLE activity:        {df['total_clicks'].isna().sum()} ({df['total_clicks'].isna().mean()*100:.1f}%)")

df[assess_cols] = df[assess_cols].fillna(0)
df[vle_cols] = df[vle_cols].fillna(0)

remaining = df.select_dtypes(include=[np.number]).isnull().sum()
remaining = remaining[remaining > 0]
print(f"\nRemaining NaN in numeric columns: {len(remaining)}")

## 5. Feature Engineering

We create derived features based on EDA insights. Each feature is tagged with whether it is available in an early-warning scenario.

In [ ]:
# Engagement intensity normalized by module length
df['clicks_per_day'] = df['total_clicks'] / df['module_presentation_length']

# Activity diversity relative to module length
df['activity_diversity'] = df['distinct_pages'] / df['module_presentation_length']

# Engagement trend: second-half / first-half clicks
df['engagement_trend'] = df['clicks_second_half'] / df['clicks_first_half'].replace(0, np.nan)
df['engagement_trend'] = df['engagement_trend'].fillna(0)

# Forum participation ratio
df['forum_ratio'] = df['clicks_forumng'] / df['total_clicks'].replace(0, np.nan)
df['forum_ratio'] = df['forum_ratio'].fillna(0)

# Score range
df['score_range'] = df['max_score'] - df['min_score']

print(f"Engineered 5 new features. Shape: {df.shape}")

Let us verify the merged dataset.

In [ ]:
print(f"Final merged dataframe: {df.shape[0]} rows × {df.shape[1]} columns")
print(f"Unique students: {df.id_student.nunique()}")
if PREDICTION_DAY is not None:
    print(f"\nPREDICTION DAY: {PREDICTION_DAY} — only data up to this day is included in VLE/assessment features")
else:
    print("\nRETROSPECTIVE MODE — all data included")
print(f"\nFirst 5 rows (selected columns):")
preview_cols = ['id_student','code_module','final_result','total_clicks','active_days',
                'submission_rate','engagement_trend','clicks_per_day']
df[preview_cols].head()

**Feature summary with availability tagging:**

| Feature | Available at prediction time? |
|---|---|
| Demographics (gender, age, education, IMD, disability, region) | ✓ Always available |
| num_of_prev_attempts, studied_credits | ✓ Always available |
| Module and semester encodings | ✓ Always available |
| total_clicks, active_days, distinct_pages | ✓ Filtered to PREDICTION_DAY |
| clicks_per_day, activity_diversity | ✓ Filtered to PREDICTION_DAY |
| clicks_first_half, clicks_second_half, engagement_trend | ✓ Filtered to PREDICTION_DAY |
| Per-activity clicks (oucontent, forum, quiz, etc.) | ✓ Filtered to PREDICTION_DAY |
| first_click_date, forum_ratio | ✓ Filtered to PREDICTION_DAY |
| submission_rate, num_submissions, mean_days_early, pct_late | ⚠ Only submissions before PREDICTION_DAY |
| mean_score, std_score, min_score, max_score, score_range | ⚠ Only scores submitted before PREDICTION_DAY |
| weighted_coursework, exam_score | ⚠ Only if submitted before PREDICTION_DAY |
| has_submitted_coursework, has_sat_exam | ⚠ Only submissions before PREDICTION_DAY |

When `PREDICTION_DAY` is set to an early value (e.g., day 30), very few assessment scores exist — the model must rely primarily on VLE engagement and demographics. As the prediction day moves later, more scores become available and accuracy should increase.

## 6. Encoding Categorical Variables

**Note on scaling:** Feature scaling (StandardScaler) is required for SVM and Logistic Regression but is deferred to Phase 2 — it must be fitted on training data only after the train-test split.

In [ ]:
# Ordinal encoding
education_order = ['No Formal quals', 'Lower Than A Level', 'A Level or Equivalent',
                   'HE Qualification', 'Post Graduate Qualification']
age_order = ['0-35', '35-55', '55<=']
imd_order = ['0-10%','10-20','20-30%','30-40%','40-50%','50-60%','60-70%','70-80%','80-90%','90-100%']

df['education_ord'] = pd.Categorical(df['highest_education'], categories=education_order, ordered=True).codes
df['age_ord'] = pd.Categorical(df['age_band'], categories=age_order, ordered=True).codes
df['imd_ord'] = pd.Categorical(df['imd_band'], categories=imd_order, ordered=True).codes

# Binary encoding (M=1/F=0, Y=1/N=0 — second value is implicitly 0)
df['gender_enc'] = (df['gender'] == 'M').astype(int)
df['disability_enc'] = (df['disability'] == 'Y').astype(int)

# One-hot encoding
module_dummies = pd.get_dummies(df['code_module'], prefix='mod').astype(int)
region_dummies = pd.get_dummies(df['region'], prefix='reg').astype(int)
df = pd.concat([df, module_dummies, region_dummies], axis=1)

# Semester (B=February=0, J=October=1)
df['semester'] = df['code_presentation'].str.contains('J').astype(int)

# Target encoding
df['target'] = df['final_result'].map({'Distinction': 0, 'Pass': 1, 'Fail': 2, 'Withdrawn': 3})
df['target_binary'] = df['final_result'].isin(['Fail','Withdrawn']).astype(int)

print("Encoding complete.")
print(f"\nMulti-class target:\n{df.target.value_counts().sort_index()}")
print(f"\nBinary target:\n{df.target_binary.value_counts()}")

**Encoding decisions:**
- **Ordinal** for education (0-4), age (0-2), IMD (0-9) — natural ranking preserved.
- **Binary** for gender, disability, semester — single column sufficient for two values.
- **One-hot** for module (7) and region (13) — no natural ordering.
- **Two targets** created: 4-class and binary (Success vs At-Risk; nearly balanced: ~47% vs ~53%).

**Phase 2 notes:** Feature scaling and imputation should be recomputed on training data only after `GroupShuffleSplit` by `id_student`.